In [1]:
#kernel thesis clean4
import pickle
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_pickle("screw_data_s02-v2_identical-to-v1.pkl")
df.head()

,time_values,torque_values,angle_values,gradient_values,step_values,class_values,workpiece_location,workpiece_usage,workpiece_result,scenario_condition,scenario_exception
0,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.067, 0.077, 0.126, 0.087, 0.089, 0.097, 0.0...","[0.5, 1.25, 2.25, 3.75, 5.0, 6.25, 7.5, 8.75, ...","[0.0, 0.0, 0.0298, 0.0214, 0.0064, 0.0023, -0....","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,0,OK,normal,0
1,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.005, 0.008, 0.061, 0.069, 0.104, 0.124, 0....","[0.0, 0.25, 0.75, 1.5, 2.5, 4.0, 5.25, 6.5, 7....","[0.0, 0.0, 0.0, 0.0, 0.0287, 0.0282, 0.0091, 0...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,0,OK,normal,0
2,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, 0.01, 0.039, 0.099, 0.102, 0.087, 0.0...","[0.0, 0.5, 1.25, 2.25, 3.5, 4.75, 6.0, 7.5, 8....","[0.0, 0.0, 0.0, 0.0196, 0.0223, 0.0211, 0.0096...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,1,OK,normal,0
3,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.081, 0.059, 0.12, 0.077, 0.043, 0.059, 0.06...","[0.75, 1.75, 2.75, 4.0, 5.25, 6.5, 7.75, 9.25,...","[0.0, 0.0, 0.0261, 0.0207, 0.0, -0.0036, -0.00...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,1,OK,normal,0
4,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, -0.0012, 0.0006, 0.0024, 0.0042, 0.00...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 1.5, 2.5, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.025...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,2,OK,normal,0


In [26]:
x_data = np.array(df['torque_values'].tolist())
y_data = np.array(df['class_values'].tolist())

le = LabelEncoder()
y_encoded = le.fit_transform(y_data)

X_train_full, X_test, y_train_full, y_test = train_test_split(x_data, y_encoded, test_size=0.2, stratify=y_encoded,random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full,y_train_full,test_size=0.25,stratify=y_train_full,random_state=42)

X_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(-1)
X_val   = torch.tensor(X_val, dtype=torch.float32).unsqueeze(-1)
X_test  = torch.tensor(X_test, dtype=torch.float32).unsqueeze(-1)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

X_mark_enc_train = torch.ones((X_train.shape[0], X_train.shape[1]), dtype=torch.float32)
X_mark_enc_test  = torch.ones((X_test.shape[0], X_test.shape[1]), dtype=torch.float32)
X_mark_enc_val   = torch.ones((X_val.shape[0], X_val.shape[1]), dtype=torch.float32)

train_dataset = TensorDataset(X_train, X_mark_enc_train, y_train)
val_dataset   = TensorDataset(X_val, X_mark_enc_val, y_val)
test_dataset  = TensorDataset(X_test, X_mark_enc_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: torch.Size([7500, 800, 1]) torch.Size([7500])
Validation: torch.Size([2500, 800, 1]) torch.Size([2500])
Test: torch.Size([2500, 800, 1]) torch.Size([2500])


In [2]:
import sys
import os
times_net_path = os.path.abspath("./Time-Series-Library")
sys.path.append(times_net_path)

from models import TimesNet
from types import SimpleNamespace

In [3]:
#import sys
#import os
#sys.path.append(r"C:\Users\Patrick\Time-Series-Library")
#from models import TimesNet
#from types import SimpleNamespace


configs = SimpleNamespace(
    task_name="classification",
    seq_len=800,
    pred_len=0,
    enc_in=1,
    num_class=8,
    d_model=32, #64
    d_ff=64, #128
    e_layers=2, #3
    dropout=0.1,
    top_k=3, #5
    num_kernels=3,
    label_len=0,
    embed = "timeF",
    freq = "h",
    activation = "gelu"
)
model = TimesNet.Model(configs)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Model(
  (model): ModuleList(
    (0-1): 2 x TimesBlock(
      (conv): Sequential(
        (0): Inception_Block_V1(
          (kernels): ModuleList(
            (0): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1))
            (1): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (2): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
          )
        )
        (1): GELU(approximate='none')
        (2): Inception_Block_V1(
          (kernels): ModuleList(
            (0): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1))
            (1): Conv2d(64, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (2): Conv2d(64, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
          )
        )
      )
    )
  )
  (enc_embedding): DataEmbedding(
    (value_embedding): TokenEmbedding(
      (tokenConv): Conv1d(1, 32, kernel_size=(3,), stride=(1,), padding=(1,), bias=False, padding_mode=circular)
    )
    (position_embedding): P

In [4]:
class EarlyStopper:
    def __init__(self, patience=1, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = float('inf')

    def early_stop(self, validation_loss):
        if validation_loss < self.min_validation_loss:
            self.min_validation_loss = validation_loss
            self.counter = 0
        elif validation_loss > (self.min_validation_loss + self.min_delta):
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

In [30]:
from sklearn.metrics import f1_score
import math
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

earlystop = EarlyStopper(patience=5, min_delta=0.001)

epochs = 50

train_losses = []
val_losses = []
val_f1_scores = []

best_val_f1 = -np.inf
best_model_state = None

for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    for X_batch, X_mark_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        X_mark_batch = X_mark_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch, X_mark_batch, None, None)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    model.eval()

    val_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_val_batch, X_mark_val_batch, y_val_batch in val_loader:
            X_val_batch = X_val_batch.to(device)
            X_mark_val_batch = X_mark_val_batch.to(device)
            y_val_batch = y_val_batch.to(device)

            outputs = model(X_val_batch, X_mark_val_batch, None, None)
            loss = criterion(outputs, y_val_batch)

            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_val_batch.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    val_f1 = f1_score(all_labels, all_preds, average="macro")
    val_f1_scores.append(val_f1)
    scheduler.step(avg_val_loss)

    #best model speichern für test
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = model.state_dict()

    if earlystop.early_stop(avg_val_loss):
        print("Early stopping triggered")
        break

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val F1 Score: {val_f1:.4f}")


Epoch 1/50, Train Loss: 2.3726, Val Loss: 2.0046, Val F1 Score: 0.1619
Epoch 2/50, Train Loss: 1.8881, Val Loss: 1.8217, Val F1 Score: 0.2392
Epoch 3/50, Train Loss: 1.8009, Val Loss: 1.7762, Val F1 Score: 0.2387
Epoch 4/50, Train Loss: 1.7412, Val Loss: 1.7745, Val F1 Score: 0.2051
Epoch 5/50, Train Loss: 1.7183, Val Loss: 1.7442, Val F1 Score: 0.2445
Epoch 6/50, Train Loss: 1.7031, Val Loss: 1.7054, Val F1 Score: 0.2941
Epoch 7/50, Train Loss: 1.7036, Val Loss: 1.7331, Val F1 Score: 0.2774
Epoch 8/50, Train Loss: 1.6785, Val Loss: 1.7621, Val F1 Score: 0.2743
Epoch 9/50, Train Loss: 1.6668, Val Loss: 1.7131, Val F1 Score: 0.2524
Epoch 10/50, Train Loss: 1.6551, Val Loss: 1.6944, Val F1 Score: 0.2581
Epoch 11/50, Train Loss: 1.6448, Val Loss: 1.7006, Val F1 Score: 0.2763
Epoch 12/50, Train Loss: 1.6352, Val Loss: 1.7042, Val F1 Score: 0.2633
Epoch 13/50, Train Loss: 1.6141, Val Loss: 1.6930, Val F1 Score: 0.2829
Epoch 14/50, Train Loss: 1.6214, Val Loss: 1.6773, Val F1 Score: 0.2928
E

In [32]:
model.load_state_dict(best_model_state)
#torch.save(best_model_state, "best_inception_model.pth")
model.eval()

test_preds = []
test_labels = []
test_loss = 0.0

with torch.no_grad():
    for X_test_batch, X_mark_enc_test_batch, y_test_batch in test_loader:
        X_test_batch = X_test_batch.to(device)
        X_mark_enc_test_batch = X_mark_enc_test_batch.to(device)
        y_test_batch = y_test_batch.to(device)

        outputs = model(X_test_batch, X_mark_enc_test_batch, None, None)
        loss = criterion(outputs, y_test_batch)

        test_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(y_test_batch.cpu().numpy())

test_loss = test_loss / len(test_loader)
test_f1 = f1_score(test_labels, test_preds, average="macro")

print(f"Test Loss: {test_loss:.4f}")
print(f"Test F1 Macro: {test_f1:.4f}")

Test Loss: 1.6528
Test F1 Macro: 0.3016


## 3- Cross validation

In [5]:
from sklearn.metrics import f1_score
import math
x_data = np.array(df['torque_values'].tolist())[..., np.newaxis]
y_data = np.array(df['class_values'].tolist())

print("x_data shape:", x_data.shape)
print("transposed x_data shape:", x_data.shape)
le = LabelEncoder()
y_encoded = le.fit_transform(y_data)

X_train_full, X_test, y_train_full, y_test = train_test_split(x_data, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=81)

cv_scores = []

best_overall_model_state = None
best_overall_f1 = -np.inf


for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train_full)):

    print(f"Fold {fold+1}")

    X_train_fold = X_train_full[train_idx]
    y_train_fold = y_train_full[train_idx]

    X_val_fold = X_train_full[val_idx]
    y_val_fold = y_train_full[val_idx]
    X_mark_enc_train = torch.ones((X_train_fold.shape[0], X_train_fold.shape[1]), dtype=torch.float32) 
    X_mark_enc_val   = torch.ones((X_val_fold.shape[0], X_val_fold.shape[1]), dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(torch.tensor(X_train_fold, dtype=torch.float32), torch.tensor(X_mark_enc_train, dtype=torch.float32), torch.tensor(y_train_fold, dtype=torch.long)), batch_size=32, shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.tensor(X_val_fold, dtype=torch.float32), torch.tensor(X_mark_enc_val, dtype=torch.float32), torch.tensor(y_val_fold, dtype=torch.long)), batch_size=32, shuffle=False)
    model = TimesNet.Model(configs)
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    earlystop = EarlyStopper(patience=7, min_delta=0.001)

    epochs = 50

    best_val_f1 = -np.inf
    best_model_state = None


    for epoch in range(epochs):

        model.train()
        train_loss = 0.0

        for X_batch, X_mark_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            X_mark_batch = X_mark_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch, X_mark_batch, None, None)
            loss = criterion(outputs, y_batch)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for X_val_batch, X_mark_val_batch, y_val_batch in val_loader:
                X_val_batch = X_val_batch.to(device)
                X_mark_val_batch = X_mark_val_batch.to(device)
                y_val_batch = y_val_batch.to(device)

                outputs = model(X_val_batch, X_mark_val_batch, None, None)
                loss = criterion(outputs, y_val_batch)

                val_loss += loss.item()

                preds = torch.argmax(outputs, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y_val_batch.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)
        val_f1 = f1_score(all_labels, all_preds, average="macro")

        scheduler.step(avg_val_loss)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = model.state_dict()

        if earlystop.early_stop(avg_val_loss):
            print("Early stopping triggered")
            break

        print(f"Fold {fold+1}, Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val F1: {val_f1:.4f}")

    cv_scores.append(best_val_f1)
    if best_val_f1 > best_overall_f1:
        best_overall_f1 = best_val_f1
        best_overall_model_state = best_model_state


print(f"CV f1 mean avg score: {np.mean(cv_scores):.4f}, std: {np.std(cv_scores):.4f}")

x_data shape: (12500, 800, 1)
transposed x_data shape: (12500, 800, 1)
Fold 1


C:\Users\Patrick\AppData\Local\Temp\ipykernel_2384\6720445.py:35: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_loader = DataLoader(TensorDataset(torch.tensor(X_train_fold, dtype=torch.float32), torch.tensor(X_mark_enc_train, dtype=torch.float32), torch.tensor(y_train_fold, dtype=torch.long)), batch_size=32, shuffle=True)
C:\Users\Patrick\AppData\Local\Temp\ipykernel_2384\6720445.py:36: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  val_loader = DataLoader(TensorDataset(torch.tensor(X_val_fold, dtype=torch.float32), torch.tensor(X_mark_enc_val, dtype=torch.float32), torch.tensor(y_val_fold, dtype=torch.long)), batch_size=32, shuffle=False)


Fold 1, Epoch 1/50, Train Loss: 2.5808, Val Loss: 1.9348, Val F1: 0.1620
Fold 1, Epoch 2/50, Train Loss: 1.9640, Val Loss: 1.8496, Val F1: 0.2025
Fold 1, Epoch 3/50, Train Loss: 1.8589, Val Loss: 1.7933, Val F1: 0.2177
Fold 1, Epoch 4/50, Train Loss: 1.7881, Val Loss: 1.7712, Val F1: 0.2187
Fold 1, Epoch 5/50, Train Loss: 1.7546, Val Loss: 1.7668, Val F1: 0.2224
Fold 1, Epoch 6/50, Train Loss: 1.7256, Val Loss: 1.7318, Val F1: 0.2333
Fold 1, Epoch 7/50, Train Loss: 1.7146, Val Loss: 1.7163, Val F1: 0.2621
Fold 1, Epoch 8/50, Train Loss: 1.6899, Val Loss: 1.7295, Val F1: 0.2652
Fold 1, Epoch 9/50, Train Loss: 1.6771, Val Loss: 1.7417, Val F1: 0.2170
Fold 1, Epoch 10/50, Train Loss: 1.6745, Val Loss: 1.6950, Val F1: 0.2439
Fold 1, Epoch 11/50, Train Loss: 1.6479, Val Loss: 1.7069, Val F1: 0.2463
Fold 1, Epoch 12/50, Train Loss: 1.6343, Val Loss: 1.6880, Val F1: 0.2795
Fold 1, Epoch 13/50, Train Loss: 1.6195, Val Loss: 1.7235, Val F1: 0.2557
Fold 1, Epoch 14/50, Train Loss: 1.6102, Val Lo

C:\Users\Patrick\AppData\Local\Temp\ipykernel_2384\6720445.py:35: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_loader = DataLoader(TensorDataset(torch.tensor(X_train_fold, dtype=torch.float32), torch.tensor(X_mark_enc_train, dtype=torch.float32), torch.tensor(y_train_fold, dtype=torch.long)), batch_size=32, shuffle=True)
C:\Users\Patrick\AppData\Local\Temp\ipykernel_2384\6720445.py:36: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  val_loader = DataLoader(TensorDataset(torch.tensor(X_val_fold, dtype=torch.float32), torch.tensor(X_mark_enc_val, dtype=torch.float32), torch.tensor(y_val_fold, dtype=torch.long)), batch_size=32, shuffle=False)


Fold 2, Epoch 1/50, Train Loss: 2.6460, Val Loss: 1.9979, Val F1: 0.1821
Fold 2, Epoch 2/50, Train Loss: 1.9512, Val Loss: 2.0119, Val F1: 0.1304
Fold 2, Epoch 3/50, Train Loss: 1.8518, Val Loss: 1.8162, Val F1: 0.1891
Fold 2, Epoch 4/50, Train Loss: 1.7828, Val Loss: 1.7384, Val F1: 0.2524
Fold 2, Epoch 5/50, Train Loss: 1.7453, Val Loss: 1.8217, Val F1: 0.2152
Fold 2, Epoch 6/50, Train Loss: 1.7284, Val Loss: 1.7239, Val F1: 0.2220
Fold 2, Epoch 7/50, Train Loss: 1.7001, Val Loss: 1.7536, Val F1: 0.2331
Fold 2, Epoch 8/50, Train Loss: 1.6876, Val Loss: 1.7182, Val F1: 0.2606
Fold 2, Epoch 9/50, Train Loss: 1.6677, Val Loss: 1.6820, Val F1: 0.2683
Fold 2, Epoch 10/50, Train Loss: 1.6648, Val Loss: 1.7610, Val F1: 0.2723
Fold 2, Epoch 11/50, Train Loss: 1.6551, Val Loss: 1.7025, Val F1: 0.2769
Fold 2, Epoch 12/50, Train Loss: 1.6438, Val Loss: 1.7537, Val F1: 0.2362
Fold 2, Epoch 13/50, Train Loss: 1.6409, Val Loss: 1.7114, Val F1: 0.2585
Fold 2, Epoch 14/50, Train Loss: 1.5750, Val Lo

C:\Users\Patrick\AppData\Local\Temp\ipykernel_2384\6720445.py:35: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_loader = DataLoader(TensorDataset(torch.tensor(X_train_fold, dtype=torch.float32), torch.tensor(X_mark_enc_train, dtype=torch.float32), torch.tensor(y_train_fold, dtype=torch.long)), batch_size=32, shuffle=True)
C:\Users\Patrick\AppData\Local\Temp\ipykernel_2384\6720445.py:36: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  val_loader = DataLoader(TensorDataset(torch.tensor(X_val_fold, dtype=torch.float32), torch.tensor(X_mark_enc_val, dtype=torch.float32), torch.tensor(y_val_fold, dtype=torch.long)), batch_size=32, shuffle=False)


Fold 3, Epoch 1/50, Train Loss: 2.6028, Val Loss: 1.8692, Val F1: 0.1969
Fold 3, Epoch 2/50, Train Loss: 1.8976, Val Loss: 1.8519, Val F1: 0.2182
Fold 3, Epoch 3/50, Train Loss: 1.8337, Val Loss: 1.7963, Val F1: 0.2311
Fold 3, Epoch 4/50, Train Loss: 1.7784, Val Loss: 1.7710, Val F1: 0.2349
Fold 3, Epoch 5/50, Train Loss: 1.7385, Val Loss: 1.7156, Val F1: 0.2948
Fold 3, Epoch 6/50, Train Loss: 1.7089, Val Loss: 1.7083, Val F1: 0.2888
Fold 3, Epoch 7/50, Train Loss: 1.7063, Val Loss: 1.6971, Val F1: 0.2771
Fold 3, Epoch 8/50, Train Loss: 1.6794, Val Loss: 1.6893, Val F1: 0.2751
Fold 3, Epoch 9/50, Train Loss: 1.6660, Val Loss: 1.6952, Val F1: 0.2550
Fold 3, Epoch 10/50, Train Loss: 1.6629, Val Loss: 1.6891, Val F1: 0.2817
Fold 3, Epoch 11/50, Train Loss: 1.6435, Val Loss: 1.6715, Val F1: 0.2649
Fold 3, Epoch 12/50, Train Loss: 1.6473, Val Loss: 1.6709, Val F1: 0.2680
Fold 3, Epoch 13/50, Train Loss: 1.6339, Val Loss: 1.6519, Val F1: 0.2779
Fold 3, Epoch 14/50, Train Loss: 1.6221, Val Lo

In [6]:
model = TimesNet.Model(configs)
final_model = model
final_model.to(device)
final_model.load_state_dict(best_overall_model_state)
final_model.eval()
X_mark_enc_test = torch.ones((X_test.shape[0], X_test.shape[1]), dtype=torch.float32)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(X_mark_enc_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long)), batch_size=32, shuffle=False)

all_preds = []
all_labels = []
test_loss = 0.0

criterion = nn.CrossEntropyLoss()
with torch.no_grad():
    for X_test_batch, X_mark_test_batch, y_test_batch in test_loader:

        X_test_batch = X_test_batch.to(device)
        X_mark_test_batch = X_mark_test_batch.to(device)
        y_test_batch = y_test_batch.to(device)
        outputs = final_model(X_test_batch, X_mark_test_batch, None, None)
        loss = criterion(outputs, y_test_batch)
        test_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_test_batch.cpu().numpy())

test_loss = test_loss / len(test_loader)
test_f1 = f1_score(all_labels, all_preds, average="macro")
print(f"Final Test F1 Macro: {test_f1:.4f}")

C:\Users\Patrick\AppData\Local\Temp\ipykernel_2384\1206236275.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_loader = DataLoader(TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(X_mark_enc_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long)), batch_size=32, shuffle=False)


Final Test F1 Macro: 0.3228
